# Final RAG Pipeline Design — Notebook Overview

This notebook demonstrates the finalized Multi-Page RAG pipeline and documents the design choices, backend selection logic, and recommended experiments/tests.

Key points:
- Backends supported: OpenAI (remote), Ollama (local), HuggingFace (local/hosted), and a TF-IDF fallback.
- Selection priority (high-level): FORCE_OLLAMA -> OpenAI -> Ollama autodetect -> HuggingFace -> TF-IDF fallback.
- Rationale: provide portability (offline TF-IDF), high-quality results (OpenAI), and a local, low-latency option (Ollama) for development and CI.

What you'll see when running cells:
- Informational logs indicating which backend was chosen, e.g. 'Using OpenAI embeddings via FAISS.' or 'Falling back to TF-IDF vector store.'
- When an LLM runs, outputs include a Sources list derived from chunk metadata so answers are traceable.

Suggested experiments (in order):
1. Run the notebook with no special env vars — observe whether HuggingFace or TF-IDF is used depending on your environment.
2. Set `OPENAI_API_KEY` and re-run to see OpenAI selected (if available).
3. Start a local Ollama instance and set `FORCE_OLLAMA=1` to test local model behavior.
4. Run the chunking unit tests (see tests/) to validate deterministic behavior with seeds.

Notes:
- Cells later in this notebook implement the same helper functions used by the CLI: fetching, cleaning, randomized chunking, vector store construction, retrieval search, and RetrievalQA.

## 0 — Install dependencies

Install required packages in your environment. If you're running this notebook locally, run:

```bash
pip install -r requirements.txt
# or individually:
# pip install langchain faiss-cpu openai requests beautifulsoup4 scikit-learn numpy sentence-transformers transformers
```

If you don't want to install heavy packages, the TF-IDF fallback will still work.

In [9]:
# 1 — Imports & Configuration
import os, re, random, logging
from typing import List, Optional, Tuple, Any
import requests
from bs4 import BeautifulSoup

# Optional libraries; the notebook gracefully falls back if they are not installed.
try:
    from langchain.docstore.document import Document
    from langchain.embeddings import OpenAIEmbeddings, HuggingFaceEmbeddings
    from langchain.vectorstores import FAISS
    from langchain.chains import RetrievalQA
    from langchain.llms import OpenAI, HuggingFaceHub
    HAS_LANGCHAIN = True
except Exception as e:
    print('LangChain not available or partial:', e)
    HAS_LANGCHAIN = False

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    import numpy as np
    HAS_SKLEARN = True
except Exception as e:
    print('scikit-learn not available:', e)
    HAS_SKLEARN = False

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger()

# Chunking parameters
OVERLAP = 50
MIN_CHUNK = 400
MAX_CHUNK = 600

# Default URLs (you can change these)
URL_QC = 'https://en.wikipedia.org/wiki/Quantum_computing'
URL_QML = 'https://en.wikipedia.org/wiki/Quantum_machine_learning'


In [10]:
# 2 — Fetching & Cleaning functions
from typing import Tuple
def fetch_html(url: str) -> str:
    resp = requests.get(url, headers={"User-Agent": "multi-page-rag/1.0"})
    resp.raise_for_status()
    return resp.text

def clean_wikipedia_html(html: str) -> Tuple[str, str]:
    soup = BeautifulSoup(html, 'html.parser')
    title_tag = soup.find('h1', id='firstHeading')
    title = title_tag.get_text(strip=True) if title_tag else ''
    content = soup.find('div', id='mw-content-text') or soup.find('article') or soup
    for tag in content.find_all(['table','style','script','sup','img','aside']):
        tag.decompose()
    stop_headings = {'References','External links','See also','Further reading'}
    paragraphs = []
    for elem in content.find_all(['h2','h3','p','li']):
        if elem.name in ('h2','h3'):
            heading_text = re.sub(r"\[.*?\]", '', elem.get_text(' ', strip=True))
            if any(h in heading_text for h in stop_headings):
                break
            continue
        text = elem.get_text(' ', strip=True)
        if not text:
            continue
        text = re.sub(r"\[\d+\]", '', text)
        text = re.sub(r"\[citation needed\]", '', text, flags=re.IGNORECASE)
        text = re.sub(r"\[.*?\]", '', text)
        text = re.sub(r"\s+", ' ', text).strip()
        if len(text) > 30:
            paragraphs.append(text)
    cleaned = '\n\n'.join(paragraphs)
    return title, cleaned

# Quick preview
for url in [URL_QC, URL_QML]:
    print('Fetching and cleaning:', url)
    html = fetch_html(url)
    title, cleaned = clean_wikipedia_html(html)
    print('\nTitle:', title)
    print('\nPreview (first 400 chars):\n', cleaned[:400].replace('\n',' '))
    print('\n---\n')


Fetching and cleaning: https://en.wikipedia.org/wiki/Quantum_computing

Title: Quantum computing

Preview (first 400 chars):
 A quantum computer is a (real or theoretical) computer that uses quantum mechanical phenomena in an essential way: it exploits superposed and entangled states , and the intrinsically non-deterministic outcomes of quantum measurements , as features of its computation. Quantum computers can be viewed as sampling from quantum systems that evolve in ways classically described as operating on an enormo

---

Fetching and cleaning: https://en.wikipedia.org/wiki/Quantum_machine_learning

Title: Quantum computing

Preview (first 400 chars):
 A quantum computer is a (real or theoretical) computer that uses quantum mechanical phenomena in an essential way: it exploits superposed and entangled states , and the intrinsically non-deterministic outcomes of quantum measurements , as features of its computation. Quantum computers can be viewed as sampling from quantum systems t

In [11]:
# 3 — Randomized overlapping chunking
import random
def randomized_chunks(text: str, min_size=MIN_CHUNK, max_size=MAX_CHUNK, overlap=OVERLAP, seed: Optional[int] = None):
    if seed is not None:
        random.seed(seed)
    pos = 0
    n = len(text)
    chunks = []
    while pos < n:
        size = random.randint(min_size, max_size)
        chunk = text[pos: pos + size].strip()
        if not chunk:
            break
        chunks.append(chunk)
        pos += max(1, size - overlap)
        if len(chunks) > 10000:
            break
    return chunks

# Example: chunk QC cleaned text
_, cleaned_qc = clean_wikipedia_html(fetch_html(URL_QC))
chunks_qc = randomized_chunks(cleaned_qc, seed=123)
print('QC chunks:', len(chunks_qc))
print('Example chunk length distribution: min', min(len(c) for c in chunks_qc), 'max', max(len(c) for c in chunks_qc))
print('\nFirst chunk preview:\n', chunks_qc[0][:500])


QC chunks: 103
Example chunk length distribution: min 192 max 598

First chunk preview:
 A quantum computer is a (real or theoretical) computer that uses quantum mechanical phenomena in an essential way: it exploits superposed and entangled states , and the intrinsically non-deterministic outcomes of quantum measurements , as features of its computation. Quantum computers can be viewed as sampling from quantum systems that evolve in ways classically described as operating on an enormous number of


In [12]:
# 4 — Build documents (chunks with metadata)
from dataclasses import dataclass
def build_docs_for_urls(urls: List[str], seed: Optional[int] = None):
    docs = []
    for url in urls:
        html = fetch_html(url)
        title, cleaned = clean_wikipedia_html(html)
        chunks = randomized_chunks(cleaned, seed=seed)
        for i, c in enumerate(chunks):
            if 'Document' in globals():
                docs.append(Document(page_content=c, metadata={'source': url, 'title': title, 'chunk_index': i}))
            else:
                docs.append({'page_content': c, 'metadata': {'source': url, 'title': title, 'chunk_index': i}})
    return docs

urls = [URL_QC, URL_QML]
docs = build_docs_for_urls(urls, seed=42)
print('Total docs (chunks):', len(docs))
print('Sample metadata:', docs[0].metadata if hasattr(docs[0], 'metadata') else docs[0]['metadata'])


Total docs (chunks): 182
Sample metadata: {'source': 'https://en.wikipedia.org/wiki/Quantum_computing', 'title': 'Quantum computing', 'chunk_index': 0}


In [13]:
# 5 — Build vector store (OpenAI / HuggingFace / TF-IDF fallback)
def build_vectorstore(docs: List[Any]):
    # Try OpenAI embeddings via LangChain
    if HAS_LANGCHAIN and os.getenv('OPENAI_API_KEY'):
        try:
            emb = OpenAIEmbeddings()
            store = FAISS.from_documents(docs, emb)
            return store, 'faiss-openai'
        except Exception as e:
            print('OpenAI FAISS build failed:', e)
    # Try HuggingFace via LangChain
    if HAS_LANGCHAIN:
        try:
            emb = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
            store = FAISS.from_documents(docs, emb)
            return store, 'faiss-hf'
        except Exception as e:
            print('HuggingFace FAISS build failed:', e)
    # Fallback to TF-IDF
    if HAS_SKLEARN:
        class SimpleFallbackVectorStore:
            def __init__(self, docs):
                self.docs = docs
                texts = [d.page_content if hasattr(d, 'page_content') else d['page_content'] for d in docs]
                self.tfidf = TfidfVectorizer().fit(texts + ['placeholder'])
                self.vectors = self.tfidf.transform(texts).toarray()
            def search(self, query, k=3):
                qv = self.tfidf.transform([query]).toarray()[0]
                sims = (self.vectors @ qv) / ((np.linalg.norm(self.vectors, axis=1) * (np.linalg.norm(qv) + 1e-9)) + 1e-9)
                idxs = sims.argsort()[::-1][:k]
                return [(self.docs[i], float(sims[i])) for i in idxs]
        return SimpleFallbackVectorStore(docs), 'tfidf'
    raise RuntimeError('No vector store available')

store, stype = build_vectorstore(docs)
print('Built vector store type:', stype)


2025-10-05 07:40:38,643 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2025-10-05 07:40:38,643 INFO Retrying request to /embeddings in 0.465253 seconds
2025-10-05 07:40:38,643 INFO Retrying request to /embeddings in 0.465253 seconds
2025-10-05 07:40:39,722 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2025-10-05 07:40:39,724 INFO Retrying request to /embeddings in 0.841079 seconds
2025-10-05 07:40:39,722 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2025-10-05 07:40:39,724 INFO Retrying request to /embeddings in 0.841079 seconds
2025-10-05 07:40:41,607 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2025-10-05 07:40:41,623 INFO Use pytorch device_name: cpu
2025-10-05 07:40:41,623 INFO Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
2025-10-05 07:40:41,607 INFO HTTP Request

OpenAI FAISS build failed: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
Built vector store type: faiss-hf
Built vector store type: faiss-hf


In [14]:
# 6 — Similarity search function
import numpy as np

def search_db(vector_store, query: str, k: int = 3, filter_source: Optional[str] = None):
    if hasattr(vector_store, 'search') and not hasattr(vector_store, 'similarity_search_with_score'):
        docs_and_scores = vector_store.search(query, k * 3)
    else:
        docs_and_scores = vector_store.similarity_search_with_score(query, k * 3)
    filtered = []
    for doc, score in docs_and_scores:
        meta = doc.metadata if hasattr(doc, 'metadata') else doc.get('metadata', {})
        src = meta.get('source') if meta else None
        if filter_source and src and filter_source not in src:
            continue
        filtered.append((doc, score))
        if len(filtered) >= k:
            break
    out = []
    for doc, score in filtered:
        meta = doc.metadata if hasattr(doc, 'metadata') else doc.get('metadata', {})
        out.append({'chunk': doc.page_content, 'score': float(score), 'source': meta.get('source')})
    return out

# Test queries
queries = [
    'what are Quantum neural networks?',
    'What is the basic unit of information in quantum computing?'
]
for q in queries:
    print("\nQuery:", q)
    hits = search_db(store, q, k=3)
    for i, h in enumerate(hits, 1):
        chunk_preview = h["chunk"][:200].replace("\n", " ")
        print(f"HIT {i}: score={h['score']:.4f} source={h['source']} chunk_preview={chunk_preview}…")




Query: what are Quantum neural networks?
HIT 1: score=0.3510 source=https://en.wikipedia.org/wiki/Quantum_machine_learning chunk_preview=e, and, while the protocol relies on a universal quantum computer, under mild assumptions it can be embedded on contemporary quantum annealing hardware.  Quantum analogues or generalizations of classi…
HIT 2: score=0.5880 source=https://en.wikipedia.org/wiki/Quantum_machine_learning chunk_preview=ertain physical systems and learning systems, in particular neural networks. For example, some mathematical and numerical techniques from quantum physics are applicable to classical deep learning and …
HIT 3: score=0.5891 source=https://en.wikipedia.org/wiki/Quantum_machine_learning chunk_preview=more quantum convolutional filters make up a quantum convolutional neural network (QCNN), and each of these filters transforms input data using a quantum circuit that can be created in an organized or…

Query: What is the basic unit of information in quantum computi

In [15]:
# 7 — RetrievalQA wrapper (LangChain if available, else simple concatenation)
def build_retrieval_qa(vector_store):
    if HAS_LANGCHAIN:
        try:
            if os.getenv('OPENAI_API_KEY'):
                llm = OpenAI(temperature=0)
            else:
                llm = HuggingFaceHub(repo_id='tiiuae/falcon-7b-instruct', model_kwargs={'temperature': 0})
            retriever = vector_store.as_retriever(search_kwargs={'k': 4})
            qa = RetrievalQA.from_chain_type(llm=llm, chain_type='stuff', retriever=retriever, return_source_documents=True)
            return qa, 'langchain'
        except Exception as e:
            print('LangChain RetrievalQA construction failed:', e)
    def simple_qa(query: str, k: int = 3, filter_source: Optional[str] = None):
        hits = search_db(vector_store, query, k=k, filter_source=filter_source)
        if not hits:
            return {'answer': "I don't know based on the provided documents.", 'sources': []}
        parts = [h['chunk'] for h in hits]
        sources = [h['source'] for h in hits if h.get('source')]
        return {'answer': '\n\n'.join(parts), 'sources': list(dict.fromkeys(sources))}
    return simple_qa, 'simple'

qa, qatype = build_retrieval_qa(store)
print('QA type:', qatype)

# Example unrestricted retrieval
for q in queries:
    print('\nQUESTION:', q)
    if qatype == 'langchain':
        try:
            res = qa.run(q)
            print('LLM answer:\n', res)
        except Exception as e:
            print('LLM run failed:', e)
    else:
        res = qa(q, k=3)
        print('Answer (concatenated):\n', res['answer'][:800])
        print('Sources:', res['sources'])


QA type: langchain

QUESTION: what are Quantum neural networks?
LLM run failed: `run` not supported when there is not exactly one output key. Got ['result', 'source_documents'].

QUESTION: What is the basic unit of information in quantum computing?
LLM run failed: `run` not supported when there is not exactly one output key. Got ['result', 'source_documents'].


In [16]:
# 8 — Page-specific QA wrappers and comparison
def ask_QC(query: str):
    if qatype == 'langchain':
        print('LangChain retriever — page-specific filtering not implemented in this simple demo')
        return None
    else:
        return qa(query, k=3, filter_source=URL_QC)

def ask_QML(query: str):
    if qatype == 'langchain':
        print('LangChain retriever — page-specific filtering not implemented in this simple demo')
        return None
    else:
        return qa(query, k=3, filter_source=URL_QML)

sample_q = 'what are Quantum neural networks?'
print('Unrestricted:')
print(qa(sample_q) if qatype != 'langchain' else 'See LangChain result above')
print('\nOnly QC page:')
print(ask_QC(sample_q))
print('\nOnly QML page:')
print(ask_QML(sample_q))


Unrestricted:
See LangChain result above

Only QC page:
LangChain retriever — page-specific filtering not implemented in this simple demo
None

Only QML page:
LangChain retriever — page-specific filtering not implemented in this simple demo
None


## 9 — Observations (to fill after running experiments)

- **Readability / conciseness**: Compare LLM answer vs concatenated chunks.
- **Combining info**: Note if the LLM merges facts from both pages usefully.
- **Citations**: Check the `Sources:` returned (or printed) — are they correct?

Fill these in with concrete examples from the runs above.